# GNN Explainability
This notebook demonstrates how the GNNExplainer extracts the top-K contributing neighbor nodes that influenced a fault prediction.

In [ ]:
import sys
sys.path.append('..')

import torch
import networkx as nx
import matplotlib.pyplot as plt
from backend.app.data.factory_graph import build_synthetic_factory_graph, convert_to_pyg_heterodata
from backend.app.models.gnn_layer import HeteroEquipmentGNN, FaultPredictionHead
from backend.app.data.graph_schema import get_edge_types
from backend.app.eval.explainer import GraphFaultExplainer

import warnings
warnings.filterwarnings('ignore')

## 1. Setup Models and Mock Graph

In [ ]:
gnn = HeteroEquipmentGNN(hidden_channels=128, out_channels=64, edge_types=get_edge_types())
head = FaultPredictionHead(in_channels=64)

gnn.eval()
head.eval()

nx_graph = build_synthetic_factory_graph(num_lines=2, machines_per_line=3)
data = convert_to_pyg_heterodata(nx_graph, torch.randn((6, 256)))

with torch.no_grad():
    emb = gnn(data.x_dict, data.edge_index_dict)
    logits = head(emb['machine'])
    probs = torch.sigmoid(logits)
    print("Probabilities:", probs.squeeze().tolist())

## 2. Run Explainer

In [ ]:
explainer = GraphFaultExplainer(gnn, head)

# Explain prediction for machine node 0
explanations = explainer.explain_prediction(data, node_index=0, top_k=5)

print("Top contributing neighbors for Machine 0:")
for exp in explanations:
    print(f"- {exp.node_type} {exp.node_id} (Importance: {exp.importance_score:.4f})")